In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/CXR")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2
%matplotlib inline

print("PROJECT_ROOT:", PROJECT_ROOT)

  
from src.download_dataset import download_dataset
from src.runtime import setup_tensorflow_runtime
from src.config import RAW_DIR
print(RAW_DIR)

setup_tensorflow_runtime()
download_dataset()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PROJECT_ROOT: /workspace/CXR
/workspace/CXR/data/raw
[WARN] GPU memory growth beállítási hiba: cannot access local variable 'gpus' where it is not associated with a value
TF version: 2.14.0
GPUs:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
CPU cores:  120
CUDA_VISIBLE_DEVICES = None
TF version = 2.14.0
Built with CUDA = True
GPUs = [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Logical GPUs = [LogicalDevice(name='/device:GPU:0', device_type='GPU')]
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU matmul done in 0.06376981735229492 sec
[SKIP] Dataset already exists, no download needed.


* Preprocess images

In [5]:
# 2. QC pipeline (csak egyszer!)
from src.qc_dataset import run_dataset_qc
from src.qc_preprocessing import run_preprocessing_qc
from src.config import OUTPUT_DIR

qc_result = run_dataset_qc(
    root_dir=RAW_DIR,
    out_dir=OUTPUT_DIR / "dataset_qc",
)

run_preprocessing_qc()

# 3. dataset load
from src.dataloader import build_datasets_from_split_csvs, inspect_batch
from src.config import OUTPUT_DIR, SPLITS_DIR

train_ds, val_ds, test_ds, *_ = build_datasets_from_split_csvs(SPLITS_DIR)

inspect_batch(train_ds)

/usr/lib/python3.11/importlib/__init__.py:169: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  _bootstrap._exec(spec, module)


AttributeError: module 'numpy.random' has no attribute 'mtrand'

- Training

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path("/workspace/CXR")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.runtime import setup_tensorflow_runtime
 
setup_tensorflow_runtime()

from src.train import run_training, run_multiple_models
from src.evaluate import run_evaluation
from src.compare_models import run_model_comparison
from src.explainability import run_explainability_qc
from src.config import MODELS_DIR, SPLITS_DIR, OUTPUT_DIR

result = run_training(
    split_dir=SPLITS_DIR,
    out_dir=MODELS_DIR,
    model_name="resnet50",      # "vgg16", "efficientnetb0", "baseline_cnn"
    pretrained=True,
    do_fine_tuning=False,
    epochs_head=8,
)

result = run_training(
    split_dir=SPLITS_DIR,
    out_dir=MODELS_DIR,
    model_name="efficientnetb0",
    pretrained=True,
    do_fine_tuning=True,
    fine_tune_fraction=0.20,
    epochs_head=6,
    epochs_finetune=4,
    learning_rate_head=1e-3,
    learning_rate_finetune=1e-5,
)

comparison_df = run_multiple_models(
    split_dir=SPLITS_DIR,
    out_dir=MODELS_DIR,
    model_names=["resnet50", "vgg16", "efficientnetb0", "baseline_cnn"],
    pretrained=True,
    do_fine_tuning=False,
    epochs_head=8,
)
comparison_df


In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path("/workspace/CXR")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.train import run_training, run_multiple_models
from src.evaluate import run_evaluation
from src.compare_models import run_model_comparison
from src.explainability import run_explainability_qc
from src.compare_explainability import run_compare_explainability

from src.config import MODELS_DIR, SPLITS_DIR, OUTPUT_DIR

results = run_evaluation(model_path = MODELS_DIR / "baseline_cnn" / "best_model.keras", split_dir = SPLITS_DIR)
results = run_evaluation(model_path = MODELS_DIR / "resnet50" / "best_model.keras", split_dir = SPLITS_DIR)
results = run_evaluation(model_path = MODELS_DIR / "vgg16" / "best_model.keras", split_dir = SPLITS_DIR)
results = run_evaluation(model_path = MODELS_DIR / "efficientnetb0" / "best_model.keras", split_dir = SPLITS_DIR)

#results = run_evaluation(
#    model_path=PROJECT_DIR / "outputs" / "training" / "resnet50" / "best_model.keras",
#    split_dir=PROJECT_DIR / "outputs" / "dataset_qc" / "splits",
#    out_dir=PROJECT_DIR / "outputs" / "evaluation" / "resnet50",
#)

compare_df = run_model_comparison(
    model_names=["resnet50", "vgg16", "efficientnetb0", "baseline_cnn"],
    models_dir=MODELS_DIR,
    split_dir=SPLITS_DIR,
    out_dir=PROJECT_DIR / "outputs" / "model_comparison",
)


result = run_explainability_qc(
    model_path=MODELS_DIR / "efficientnetb0" / "best_model.keras",
    split_dir=SPLITS_DIR,
    out_dir=OUTPUT_DIR / "figures" / "explainability" / "efficientnetb0",
)

result = run_explainability_qc(
    model_name="vgg16",
    split_dir=SPLITS_DIR,
    n_correct=2,
    n_incorrect=8,
    n_random=2,
)

run_compare_explainability(
    model_names=["resnet50", "vgg16", "efficientnetb0"],
    n_examples=6,
)

